# NPS Alert Data Collection

Notebook goal: retrieve a small sample of current National Park Service alerts from the official NPS API.

Confirm that API key works, alerts endpoint returns data, available fields match the project needs, and alert text appears suitable for traveler-impact classification.

This notebook intentionally requests only five alerts. Full data collection and pagination will be added after this initial API check.

NPS API documentation:  
https://www.nps.gov/subjects/developer/api-documentation.htm


## 1. Add the API key to Google Colab Secrets




In [ ]:
from google.colab import userdata

NPS_API_KEY = userdata.get("NPS_API_KEY")

if not NPS_API_KEY:
    raise ValueError(
        "NPS_API_KEY was not found. Add it to Google Colab Secrets "
        "and enable notebook access."
    )

print("API key loaded successfully.")


API key loaded successfully.


## 2. Import packages and define the alerts endpoint

The first request will retrieve only five alerts so that the connection and response format can be checked before collecting more data.


In [ ]:
import json
import requests
import pandas as pd

BASE_URL = "https://developer.nps.gov/api/v1/alerts"

headers = {
    "X-Api-Key": NPS_API_KEY
}

params = {
    "limit": 5,
    "start": 0
}

print("Packages imported and request settings created.")


Packages imported and request settings created.


## 3. Make the initial API request



In [ ]:
try:
    response = requests.get(
        BASE_URL,
        headers=headers,
        params=params,
        timeout=30
    )

    print("Status code:", response.status_code)
    print("Request URL:", response.url)

    response.raise_for_status()

except requests.exceptions.Timeout as exc:
    raise RuntimeError("The NPS API request timed out.") from exc

except requests.exceptions.HTTPError as exc:
    raise RuntimeError(
        f"The NPS API returned an HTTP error: {response.status_code}\n"
        f"Response preview: {response.text[:500]}"
    ) from exc

except requests.exceptions.RequestException as exc:
    raise RuntimeError(f"The NPS API request failed: {exc}") from exc


Status code: 200
Request URL: https://developer.nps.gov/api/v1/alerts?limit=5&start=0


## 4. Inspect the top-level response

This step checks the overall response structure and the number of alerts reported by the API.


In [ ]:
payload = response.json()

print("Top-level keys:", list(payload.keys()))
print("Reported total alerts:", payload.get("total"))
print("Number returned in this request:", len(payload.get("data", [])))


Top-level keys: ['total', 'limit', 'start', 'data']
Reported total alerts: 625
Number returned in this request: 5


## 5. Inspect the fields in the first alert

The project should use the fields that the API actually returns rather than assuming that every proposed field is available.


In [ ]:
alerts = payload.get("data", [])

if alerts:
    first_alert = alerts[0]

    print("Fields in one alert:")
    for field in first_alert.keys():
        print("-", field)
else:
    print("The request succeeded, but no alert records were returned.")


Fields in one alert:
- id
- url
- title
- parkCode
- description
- category
- relatedRoadEvents
- lastIndexedDate


## 6. View the complete first alert



In [ ]:
if alerts:
    print(json.dumps(alerts[0], indent=2))
else:
    print("No alert is available to display.")


{
  "id": "9C29DF02-C5E8-4008-973F-26C5B8A7FC52",
  "url": "",
  "title": "Tower of Voices Maintenance",
  "parkCode": "flni",
  "description": "The Tower of Voices is currently open and is undergoing windchime testing. Currently four (4) windchimes have been removed from the Tower for maintenance and will be replaced as soon as possible.",
  "category": "Information",
  "relatedRoadEvents": [],
  "lastIndexedDate": "2026-06-22 00:00:00.0"
}


## 7. Convert the sample to a pandas DataFrame


In [ ]:
sample_df = pd.DataFrame(alerts)

print("Shape:", sample_df.shape)
display(sample_df.head())


Shape: (5, 8)


,id,url,title,parkCode,description,category,relatedRoadEvents,lastIndexedDate
0,9C29DF02-C5E8-4008-973F-26C5B8A7FC52,,Tower of Voices Maintenance,flni,The Tower of Voices is currently open and is u...,Information,[],2026-06-22 00:00:00.0
1,17F8CB55-3003-412B-97CB-13533AC195F4,,Andrew Johnson NHS Visitor Center Closed for T...,anjo,Andrew Johnson Visitor Center will be closed M...,Park Closure,[],2026-06-22 00:00:00.0
2,2F5C3616-F71A-4358-8394-DA1A758E2F00,,River Road Closure - June 22-24,dewa,River Road will be closed between Park Headqua...,Park Closure,"[{'title': 'River Road', 'id': '2F5C3616-F71A-...",2026-06-22 00:00:00.0
3,97783AC4-B242-4BF7-8C71-37A17B987928,https://www.nps.gov/glac/planyourvisit/index.htm,Going-to-the-Sun Road is Open for 2026 Season,glac,The Going-to-the-Sun Road is fully open for th...,Information,[],2026-06-22 00:00:00.0
4,D547F69E-F5AD-4F7B-B6A5-D838C3BC05C7,,Volcano Road Closed June 22 8:30am - 4:30pm,cavo,"Due to road repair and maintenance, volcano ro...",Park Closure,[],2026-06-22 00:00:00.0


## 8. List the available columns


In [ ]:
print(sample_df.columns.tolist())


['id', 'url', 'title', 'parkCode', 'description', 'category', 'relatedRoadEvents', 'lastIndexedDate']


## 9. Display candidate project fields

This cell selects only candidate fields that are actually present, so it will not fail if the API omits one of the proposed fields.


In [ ]:
candidate_columns = [
    "id",
    "parkCode",
    "title",
    "description",
    "category",
    "url",
    "lastIndexedDate"
]

available_columns = [
    column
    for column in candidate_columns
    if column in sample_df.columns
]

print("Candidate fields available:", available_columns)

if available_columns:
    display(sample_df[available_columns])
else:
    print("None of the candidate columns were found.")


Candidate fields available: ['id', 'parkCode', 'title', 'description', 'category', 'url', 'lastIndexedDate']


,id,parkCode,title,description,category,url,lastIndexedDate
0,9C29DF02-C5E8-4008-973F-26C5B8A7FC52,flni,Tower of Voices Maintenance,The Tower of Voices is currently open and is u...,Information,,2026-06-22 00:00:00.0
1,17F8CB55-3003-412B-97CB-13533AC195F4,anjo,Andrew Johnson NHS Visitor Center Closed for T...,Andrew Johnson Visitor Center will be closed M...,Park Closure,,2026-06-22 00:00:00.0
2,2F5C3616-F71A-4358-8394-DA1A758E2F00,dewa,River Road Closure - June 22-24,River Road will be closed between Park Headqua...,Park Closure,,2026-06-22 00:00:00.0
3,97783AC4-B242-4BF7-8C71-37A17B987928,glac,Going-to-the-Sun Road is Open for 2026 Season,The Going-to-the-Sun Road is fully open for th...,Information,https://www.nps.gov/glac/planyourvisit/index.htm,2026-06-22 00:00:00.0
4,D547F69E-F5AD-4F7B-B6A5-D838C3BC05C7,cavo,Volcano Road Closed June 22 8:30am - 4:30pm,"Due to road repair and maintenance, volcano ro...",Park Closure,,2026-06-22 00:00:00.0


## 10. Read the first five alerts in a traveler-friendly format




In [ ]:
if not alerts:
    print("No alerts were returned.")

for index, alert in enumerate(alerts, start=1):
    print("=" * 80)
    print(f"ALERT {index}")
    print("Park code:", alert.get("parkCode"))
    print("NPS category:", alert.get("category"))
    print("Title:", alert.get("title"))
    print("Description:", alert.get("description"))
    print("URL:", alert.get("url"))


ALERT 1
Park code: flni
NPS category: Information
Title: Tower of Voices Maintenance
Description: The Tower of Voices is currently open and is undergoing windchime testing. Currently four (4) windchimes have been removed from the Tower for maintenance and will be replaced as soon as possible.
URL: 
ALERT 2
Park code: anjo
NPS category: Park Closure
Title: Andrew Johnson NHS Visitor Center Closed for Two Days (June 22/23).
Description: Andrew Johnson Visitor Center will be closed Monday/Tuesday, June 22/23, 2026. A Visitor Contact Station will be operational for both days at the Early Home facility (201 East Depot Street), adjacent to the main parking lot.
URL: 
ALERT 3
Park code: dewa
NPS category: Park Closure
Title: River Road Closure - June 22-24
Description: River Road will be closed between Park Headquarters and Smithfield Beach from 8 a.m. to 3:30 p.m. on June 22 - 24 for shoulder work, culvert cleaning, and other roadway maintenance. Please plan accordingly and use alternate rou

## 11. Initial API Check


1. 625 active alerts: enough active alerts to continue, five alerts returned in the test request.


Status code 200


2. The text seems like it will work for traveler-impact classification because the examples contain:

What is affected, Whether something is open or closed, The dates and times of the condition, The cause of the issue, Possible alternatives or traveler actions

3. Project motivation appears supported: “Park Closure” is too broad for trip planning because it can refer to a visitor center, road, trail, facility, or potentially an entire park. This project taxonomy would provide more actionable distinctions.

4. We will need to keep original category for descriptive analysis and comparing NPS categories with practical-impact labels but it will not be the target.

It should remain separate from the text so we can can compare model performance with and without it.

5. Some important information will need to be added: the alert endpoint returned parkCode, but not the full park name.

The next data pipeline should create a park-code lookup using the NPS parks endpoint and join the full park name to each alert.

6. Four of the first five records appear to have blank alert URLs. That means:

URL should be retained, missing URLs should be counted, URL should not be treated as essential to classification, the alert ID should be preserved as the stable record identifier

7. The data are a dated snapshot

The first alert showed: lastIndexedDate: 2026-06-22

Because these are active alerts, the dataset can change. The full collection step should save:

collection_timestamp
collection_date
Raw JSON
Cleaned CSV

That gives  a reproducible snapshot even when the live API changes.

**Taxonomy issue**: boundary between construction or maintenance and the category describing its practical consequence will require a clear rule.

For instance: River Road will be closed for shoulder work and culvert cleaning.

The cause is maintenance, but the traveler's direct consequence is a road closure. Label an alert according to its primary practical effect on the traveler, not merely the cause of the condition.

Therefore:

Road closed due to maintenance → Road, parking, or transportation
Trail closed due to repairs → Trail or area access restriction
Maintenance occurring while an attraction remains open → Facility, construction, or general information



